# Shadow Supervisor — Training Run Notebook

This notebook reproduces the training and evaluation pipeline for Shadow Supervisor.

It runs scenario generation, expert dataset generation, environment-connected RL training, hardened evaluation, and plot generation.


## 0. Setup

If running in Google Colab, set `RUN_IN_COLAB = True` and update `REPO_URL`.

If running locally from the project root, keep `RUN_IN_COLAB = False`.


In [29]:
RUN_IN_COLAB = True
REPO_URL = "https://github.com/Chandan24-cell/Shadow_Supervisor-OpenEnv.git"

print("RUN_IN_COLAB:", RUN_IN_COLAB)
print("REPO_URL:", REPO_URL)


RUN_IN_COLAB: True
REPO_URL: https://github.com/Chandan24-cell/Shadow_Supervisor-OpenEnv.git


In [30]:
import os
from pathlib import Path

if RUN_IN_COLAB:
    if not Path(PROJECT_DIR).exists():
        !git clone $REPO_URL
    os.chdir(PROJECT_DIR)

print("Current directory:", os.getcwd())
!ls


Cloning into 'Shadow_Supervisor-OpenEnv'...
remote: Enumerating objects: 145, done.
remote: Counting objects: 100% (145/145), done.
remote: Compressing objects: 100% (117/117), done.
remote: Total 145 (delta 46), reused 113 (delta 26), pack-reused 0 (from 0)
Receiving objects: 100% (145/145), 809.69 KiB | 7.04 MiB/s, done.
Resolving deltas: 100% (46/46), done.
Current directory: /content/Shadow_Supervisor-OpenEnv/Shadow_Supervisor-OpenEnv/Shadow_Supervisor-OpenEnv
agents	backups  evaluation    outputs		 scripts
app	data	 media	       pyproject.toml	 server
app.py	docs	 notebooks     README.md	 tests
assets	env	 openenv.yaml  requirements.txt  training


## 1. Install dependencies


In [31]:
!python -m pip install --upgrade pip
!pip install -r requirements.txt


## 2. Verify OpenEnv compliance


In [32]:
!python -m pip show openenv-core
!python scripts/verify_openenv_compliance.py


Name: openenv-core
Version: 0.2.3
Summary: A unified framework for reinforcement learning environments
Home-page: 
Author: 
Author-email: 
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: fastapi, fastmcp, gradio, httpx, huggingface_hub, openai, pydantic, pyyaml, requests, rich, tomli, tomli-w, typer, uvicorn, websockets
Required-by: 
openenv-core version: 0.2.3
✅ OpenEnv compliance check passed.
✅ openenv-core is installed and imported.
✅ ShadowSupervisorOpenEnv wrapper works.
✅ reset(), step(), and state() work.
✅ openenv.yaml points to the OpenEnv adapter.


## 3. Generate scenario data


In [33]:
!python training/build_real_data.py
!ls -lh data | head


✅ Shadow Supervisor data generation complete.
Generated: /content/Shadow_Supervisor-OpenEnv/Shadow_Supervisor-OpenEnv/Shadow_Supervisor-OpenEnv/data/scenarios_train.json
Generated: /content/Shadow_Supervisor-OpenEnv/Shadow_Supervisor-OpenEnv/Shadow_Supervisor-OpenEnv/data/scenarios_eval.json
Generated: /content/Shadow_Supervisor-OpenEnv/Shadow_Supervisor-OpenEnv/Shadow_Supervisor-OpenEnv/data/expert_traces.jsonl
Generated: /content/Shadow_Supervisor-OpenEnv/Shadow_Supervisor-OpenEnv/Shadow_Supervisor-OpenEnv/data/naive_traces.jsonl
Generated: /content/Shadow_Supervisor-OpenEnv/Shadow_Supervisor-OpenEnv/Shadow_Supervisor-OpenEnv/data/cautious_traces.jsonl
Generated: /content/Shadow_Supervisor-OpenEnv/Shadow_Supervisor-OpenEnv/Shadow_Supervisor-OpenEnv/data/reward_logs.jsonl
Generated: /content/Shadow_Supervisor-OpenEnv/Shadow_Supervisor-OpenEnv/Shadow_Supervisor-OpenEnv/data/policy_comparison.json

Policy comparison:
{
  "description": "Policy comparison generated from public-source-ins

## 4. Build expert dataset


In [34]:
!python training/build_expert_dataset.py


✅ SFT expert dataset built.
Input expert traces: 300
Output SFT rows: 300
Saved: /content/Shadow_Supervisor-OpenEnv/Shadow_Supervisor-OpenEnv/Shadow_Supervisor-OpenEnv/data/sft_training_data.jsonl
Saved: /content/Shadow_Supervisor-OpenEnv/Shadow_Supervisor-OpenEnv/Shadow_Supervisor-OpenEnv/data/sft_training_data_preview.json
Saved: /content/Shadow_Supervisor-OpenEnv/Shadow_Supervisor-OpenEnv/Shadow_Supervisor-OpenEnv/data/action_vocab.json


## 5. Run TRL/SFT-compatible training script

This is the fallback training path. The main training proof is the environment-connected RL run in the next step.


In [35]:
!python training/train_trl.py


✅ Tiny imitation training complete.
Expert traces used: 300
Learned plan: ['assign_research', 'assign_ops', 'assign_policy', 'assign_communication', 'audit_ops', 'audit_policy', 'cross_check_research_and_ops', 'cross_check_research_and_communication', 'select_safe_fix', 'request_message_revision']
Saved trained policy: /content/Shadow_Supervisor-OpenEnv/Shadow_Supervisor-OpenEnv/Shadow_Supervisor-OpenEnv/outputs/trained_policy.json
Saved training metrics: /content/Shadow_Supervisor-OpenEnv/Shadow_Supervisor-OpenEnv/Shadow_Supervisor-OpenEnv/outputs/training_metrics.json
Saved reward logs: /content/Shadow_Supervisor-OpenEnv/Shadow_Supervisor-OpenEnv/Shadow_Supervisor-OpenEnv/data/reward_logs.jsonl

Policy summaries:
{
  "naive_supervisor": {
    "episodes": 10,
    "avg_reward": -8.3,
    "min_reward": -8.3,
    "max_reward": -8.3,
    "success_rate": 0.0,
    "unsafe_approval_rate": 1.0,
    "avg_risk_detection_count": 0.0,
    "message_revision_rate": 0.0,
    "safe_fix_selection_rate

## 6. Run environment-connected RL training


In [ ]:
!python training/train_env_rl.py
!ls -lh outputs | grep -E 'rl_policy|training|reward|safety' || true


epoch=001 mean=-7.66 elite=-1.72 best=0.35 success=0.00 unsafe=1.00
epoch=010 mean=-2.36 elite=3.08 best=5.65 success=0.00 unsafe=1.00
epoch=020 mean=3.94 elite=11.90 best=12.65 success=0.50 unsafe=0.50
epoch=030 mean=8.81 elite=13.28 best=18.10 success=1.00 unsafe=0.00
epoch=040 mean=12.04 elite=17.35 best=17.35 success=1.00 unsafe=0.00


## 7. Run hardened evaluation


In [ ]:
!python evaluation/run_eval_hardened.py
!cat outputs/policy_comparison_hardened.csv


## 8. Generate final plots


In [ ]:
!python evaluation/generate_winning_plots.py
!ls -lh outputs/*.png


## 9. Display final policy comparison


In [ ]:
import pandas as pd

df = pd.read_csv("outputs/policy_comparison_hardened.csv")
df


## 10. Display training plots


In [ ]:
from IPython.display import Image, display

plot_files = [
    "outputs/baseline_vs_trained.png",
    "outputs/winning_rl_reward_curve.png",
    "outputs/winning_training_loss.png",
    "outputs/winning_unsafe_approval_rate.png",
    "outputs/rl_safety_curve.png",
]

for plot in plot_files:
    print("\n" + plot)
    display(Image(filename=plot))


## Final interpretation

The expected final result is:

- Naive supervisor fails.
- Training candidate partially improves but remains unsafe.
- Spam-all-actions baseline fails, proving reward is hard to game.
- RL-trained supervisor reaches high reward, 100% success, and 0% unsafe approval.
